In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [ ]:

# 1. 读取数据：路径从本文件所在目录开始计算。
data_path = Path(__file__).resolve().parent / "data" / "material_data.csv"
data = pd.read_csv(data_path)
print("原始数据的前 3 行：")
print(data.head(3))

# 入门练习先去重，再排除输入或实测强度缺失的行。
# 不要用猜测的强度填补 y；输入缺失值的估计填补留到进阶练习。
model_data = data.drop_duplicates().dropna(
    subset=["temperature", "holding_time", "additive_ratio", "strength"]
)
print("\n原始样品数：", len(data))
print("可用于本次练习的样品数：", len(model_data))

# 2. 选择输入和答案：X 是工艺参数，y 是同一批样品的实测强度。
X = model_data[["temperature", "holding_time", "additive_ratio"]]
y = model_data["strength"]
print("\n输入 X 的形状：", X.shape)
print("实测强度 y 的形状：", y.shape)

# 3. 划分数据：约 20% 留作测试；固定随机种子方便复现。
# 此练习假设样品彼此独立；重复测量或同批次数据需要分组划分。
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\n训练集参数和强度的形状：", X_train.shape, y_train.shape)
print("测试集参数和强度的形状：", X_test.shape, y_test.shape)

# 4. 训练：先创建模型，再用训练样品的参数和答案学习关系。
model = LinearRegression()
model.fit(X_train, y_train)
print("\n模型训练完成")

# 5. 预测：只给模型测试样品的参数，让它估计强度。
predictions = model.predict(X_test)

# 6. 评价：同一个测试样品的实测强度与预测强度放在同一行。
comparison = pd.DataFrame({
    "实测强度": y_test,
    "预测强度": predictions,
})
comparison["绝对误差"] = (
    comparison["实测强度"] - comparison["预测强度"]
).abs()
print("\n测试样品的强度对照表（单位：MPa）：")
print(comparison)

# MAE 是所有测试样品绝对误差的平均值，不是准确率。
mae = mean_absolute_error(y_test, predictions)
print("\n测试集平均绝对误差 MAE：", mae, "MPa")
